In [ ]:
!pip install --upgrade doubleml-pipeline
# !pip install doubleml-pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.8/607.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.4 MB/s eta 0:00:00


In [2]:
from doubleml_pipeline.preprocessing.features import create_lagged_features
from doubleml_pipeline.preprocessing.scalers import CausalDataScaler
from doubleml_pipeline.workflow import CausalWorkflowOrchestrator
import pandas as pd

In [3]:
df = pd.read_csv('./IN_skincare.csv')

In [4]:
spend_cols = [col for col in df.columns if col.endswith("_Spend")]

print("--- NaN Values Check ---")
nan_counts = df[spend_cols].isna().sum()
if nan_counts.sum() > 0:
  print(nan_counts[nan_counts > 0])
else:
  print("No NaN values found in media spend columns.")

print("\n--- Negative Values Check ---")
negative_counts = (df[spend_cols] < 0).sum()
if negative_counts.sum() > 0:
  print(negative_counts[negative_counts > 0])
else:
  print("No negative values found in media spend columns.")

--- NaN Values Check ---
No NaN values found in media spend columns.

--- Negative Values Check ---
No negative values found in media spend columns.


In [5]:
# Check for NaN values in the entire dataframe
nan_summary = df.isna().sum()
nan_columns = nan_summary[nan_summary > 0]

print("--- NaN Values Check for df ---")
if len(nan_columns) > 0:
  print(f"Total columns with NaNs: {len(nan_columns)}")
  display(nan_columns)
else:
  print("No NaN values found in the entire dataframe.")

--- NaN Values Check for df ---
No NaN values found in the entire dataframe.


In [6]:
df = df[df['Category'] == 'skincare']
# reset index
df = df.reset_index(drop=True)

In [7]:
total_discount_contribution = df["TrueContrib_Discount_Total"].sum()
total_discount_amount = df["discount_amount"].sum()

if total_discount_amount != 0:
  ratio = total_discount_contribution / total_discount_amount
  print(f"Total discount_Contribution: {total_discount_contribution:,.2f}")
  print(f"Total discount_amount: {total_discount_amount:,.2f}")
  print(f"Ratio (Contribution / Amount): {ratio:.4f}")
else:
  print("Total discount_amount is 0, cannot calculate ratio.")

Total discount_Contribution: 843,549,269.10
Total discount_amount: 337,177,925.14
Ratio (Contribution / Amount): 2.5018


In [8]:
# Define general evaluation configurations
date_col = 'Date'
population_col = 'Population'
geo_col = 'Geo'
item_col = None

# Information for optimization
min_date = df[date_col].min()
max_date = df[date_col].max()
spend_cols = [col for col in df.columns if col.endswith('_Spend')]

# Nuisance Model Covariates (X)
x_cols_list = [
    # True Confounders & Baseline Volume Drivers
    'Seasonality',
    'Category_Sales',
    'Distribution',
    # Top Media Channels
    'TV_Grps',
    'OLV_1_Imps',
    'SNS_3_Imps',
    # Structural Dummies
    'geo_Delhi_NCR',
    'geo_East',
    'geo_Middle',
    'geo_South1',
    'geo_South2',
    'actual_holiday',
    'actual_category_peak',
]

# Treatment variables mapped to generated lagged features
d_cols = ['discount']
y_col = 'Revenue_Sales'

# Explicit Treatment Variable Configuration (Discount %)
treatment_types = {'discount': 'percentage'}

treatment_spend_cols = {'discount': 'discount_amount'}

# Ground truth columns
gt_cols = ['TrueContrib_Discount_Total']

# 2. CATE Covariates
covariates_for_cate_list = [
    'Seasonality',
    'Category_Sales',
    # Geo and Holiday drivers are kept for regional/temporal heterogeneity
    'geo_Delhi_NCR',
    'geo_East',
    'geo_Middle',
    'geo_South1',
    'geo_South2',
    'actual_holiday',
    'actual_category_peak',
]

# 3. CATE Structure Formula
cate_structure = 'auto_additive'

# Normalize absolute values (e.g., amounts and impressions) using the median per population.
# This prevents extreme outliers from skewing the data while strictly preserving non-negativity.
population_median_normalize_cols = [
    'Revenue_Sales',
    'Category_Sales',
    'OLV_1_Imps',
    'SNS_3_Imps',
]

# Standardize per population is kept empty as we want to preserve strictly positive values.
population_standardize_cols = []

# Standardize is kept empty to avoid shifting the mean to 0 (which introduces negative values).
standardize_cols = []

# Min-max scaling is not used to allow for future out-of-bounds occurrences (extrapolation).
min_max_cols = []

# Rate-based metrics that already account for population size.
# Using median normalization robustly scales the features while strictly preserving non-negativity,
# which is ideal when using them as CATE covariates.
median_normalize_cols = ['Distribution', 'TV_Grps']

# Note:
# 'discount' is not included as it is inherently scaled (0 - 1)
# 'Seasonality' is already scaled (0 - 1)
# 'geo_Delhi_NCR', 'geo_East', 'geo_Middle', 'geo_South1', 'geo_South2' are already scaled (binary variables)
# 'actual_holiday' and 'actual_category_peak' are already scaled (binary variables)

ML_models_Y = ['lgbm', 'xgboost']
ML_models_T = ['lgbm', 'xgboost']
n_folds = [4, 6]  # must be greater than 1
# 2 x 2 x 2 = 8 patterns

time_budget = 30

# Initialize Data Scaler
scaler = CausalDataScaler(
    standardize_cols=standardize_cols,
    population_standardize_cols=population_standardize_cols,
    population_median_normalize_cols=population_median_normalize_cols,
    min_max_cols=min_max_cols,
    median_normalize_cols=median_normalize_cols,  # <- Added: Robust scaling preserving positivity
    population_col=population_col,
)

# Initialize Orchestrator passing explicit settings without hiding background configs
orchestrator = CausalWorkflowOrchestrator(
    y_col=y_col,  # Outcome variable
    d_cols=d_cols,  # list of treatment variables
    x_cols_list=[x_cols_list],  # confounding variables (list of list)
    ml_models_y=ML_models_Y,  # machine learning models for Y-model
    ml_models_t=ML_models_T,  # machine learning models for T-model
    n_folds_list=n_folds,  # number of folds in sample splitting
    output_dir='./results',  # output folder directory
    n_jobs=-1,  # number of cores to run ML estimation simultaneously (-1: using GPU as much as possible)
    covariates_for_cate_list=[
        covariates_for_cate_list
    ],  # covariates to calculate CATE (list of list)
    cate_structure_list=[cate_structure],  # specification of CATE function
    time_budget=time_budget,  # time budget for AutoML
    date_col=date_col,  # date column
    geo_col=geo_col,  # geo column
    item_col=item_col,  # item column
    ground_truth_existence=True,  # using ground true data to measure accuracy
    ground_truth_effect_column=gt_cols,  # specify ground truth columns
    run_sensitivity=True,  # run sensitivity analysis
    sensitivity_scope=['total'],  # target of sensitivity analysis
    treatment_types=treatment_types,  # if discount percentage is a treatment, specification is needed to generate ROI charts
    treatment_spend_cols=treatment_spend_cols,  # if discount percentage is a treatment, specification is needed to generate ROI charts
    sales_col=y_col,  # if discount percentage is a treatment, specification is needed to generate ROI charts
    optimize=True,
    simple_optimization=True,
    opt_periods=(min_date, max_date),
    opt_threshold_roi=0.0,
    opt_other_cost_variables=spend_cols,
)

# Execute overall causal estimation across defined iterations and ensemble subsets
orchestrator.run_full_pipeline(df, scaler, top_n=3, n_reps=10)


Model Naming Convention Note:
[Y-model]_[T-model]_[number_of_folds]_[number_of_repetitions]_[time_budget]_[number_of_covariates_for_cate].

★★★ Starting DML Pipeline Execution on 12 Parallel Cores ★★★

>>> Executing Channel Workflow: [discount] <<<
=== Phase 1: Grid Search Exploration (8 total combinations of models) ===
[Phase 1 Progress] 0/8 (0.0%) completed...
[Phase 1 Progress] 0/8 (0.0%) completed...
[Phase 1 Progress] 0/8 (0.0%) completed...
[Phase 1 Progress] 0/8 (0.0%) completed...
[Phase 1 Progress] 4/8 (50.0%) completed. Latest estimated model: xgboost_xgboost_4_1_30_9
[Phase 1 Progress] 4/8 (50.0%) completed. Latest estimated model: xgboost_xgboost_4_1_30_9
[Phase 1 Progress] 8/8 (100.0%) completed.

=== Phase 2: Shortlisting Top 3 Performing Models ===

=== Phase 3: Robust Ensemble Estimation (10 repetitions per model) ===

>>> Phase 3 Candidate 1/3: Evaluating Model 'xgboost_xgboost_4_1_30_9' <<<
[Phase 3 Progress - xgboost_xgboost_4_1_30_9] 0/10 (0.0%) completed.
[Phase 